# AI Customer Support Agent Pipeline
A multi-stage LLM agent:
- **Stage 1:** Read the complaint and classify the issue
- **Stage 2:** Use tools: search company policies and check order information
- **Stage 3:** Decide RESOLVE vs ESCALATE and draft a reply
- **Stage 4:** Track the case (escalated cases stay unresolved)

In [6]:
import json
import re
from datetime import datetime

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

## Company Knowledge Base, Order Data and Tools

In [7]:
COMPANY_POLICIES = """
## Refund Policy
Customers may request a refund within 30 days of delivery for unused items in original packaging.
Refunds are credited to the original payment method within 5-7 business days.
Refunds up to Rs. 10,000 can be approved by the support agent directly.
Refunds above Rs. 10,000 require manager approval and must be escalated.

## Damaged or Defective Items
If an item arrives damaged or defective, the customer must report it within 7 days of delivery.
Once reported, the customer is offered a free replacement or a full refund, including shipping charges.
No return pickup fee is charged for damaged or defective items.

## Delayed Delivery
If an order is delayed by more than 5 days beyond the expected delivery date, the customer gets a Rs. 200 goodwill coupon.
If an order is delayed by more than 10 days, the customer may cancel for a full refund.
For delayed orders, apologise and share the latest tracking status.

## Return Window
Standard returns are accepted within 30 days of delivery.
Items delivered more than 30 days ago are not eligible for return or refund, except manufacturing defects under warranty.

## Warranty and Technical Support
Electronics carry a 1-year manufacturer warranty.
For technical issues, ask the customer to restart and factory reset first.
If the problem continues, raise a warranty claim; warranty claims are handled by the technical team and must be escalated.

## Billing Issues
Duplicate charges are refunded in full within 3-5 business days after verification.
Billing disputes above Rs. 10,000 must be escalated to the finance team.

## Escalation Rules
Always escalate when: the customer mentions legal action, consumer court, or media; there is a safety risk (fire, shock, injury);
the order cannot be found; no policy clearly covers the situation; or the refund amount is above Rs. 10,000.
"""

ORDERS = {
    "ORD-1001": {"customer": "Priya Menon", "item": "Wireless Earbuds", "amount": 2499, "status": "Delivered",
                 "ordered_on": "2026-11-02", "expected_delivery": "2026-11-08", "delivered_on": "2026-11-10"},
    "ORD-1002": {"customer": "Arun Kumar", "item": "Mixer Grinder", "amount": 4999, "status": "Delivered",
                 "ordered_on": "2026-11-10", "expected_delivery": "2026-11-16", "delivered_on": "2026-11-17"},
    "ORD-1003": {"customer": "Sneha Reddy", "item": "Gaming Laptop", "amount": 68000, "status": "In Transit",
                 "ordered_on": "2026-11-01", "expected_delivery": "2026-11-09", "delivered_on": None},
    "ORD-1004": {"customer": "Karthik Raja", "item": "Running Shoes", "amount": 3299, "status": "Delivered",
                 "ordered_on": "2026-09-15", "expected_delivery": "2026-09-20", "delivered_on": "2026-09-20"},
}


def parse_json(text):
    """LLMs sometimes wrap JSON in ``` fences. Strip them and parse."""
    text = re.sub(r"```(?:json)?", "", text).strip()
    return json.loads(text[text.find("{"): text.rfind("}") + 1])


def search_policies(query, top_k=2):
    """Tool 1: simple keyword search over the company policies."""
    sections = [s.strip() for s in COMPANY_POLICIES.split("## ") if s.strip()]
    words = set(re.findall(r"[a-z]+", query.lower()))
    scored = sorted(((len(words & set(re.findall(r"[a-z]+", s.lower()))), s) for s in sections),
                    key=lambda x: x[0], reverse=True)
    results = [s for score, s in scored[:top_k] if score > 0]
    escalation = next(s for s in sections if s.startswith("Escalation Rules"))
    if escalation not in results:
        results.append(escalation)  # always include escalation rules
    return "\n\n".join("## " + r for r in results)


def check_order(order_id):
    """Tool 2: look up an order (returns None if not found)."""
    return ORDERS.get(order_id) if order_id else None

## Sample Customer Complaint

In [8]:
with open("customer_complaint.txt", "r", encoding="utf-8") as f:
    complaint = f.read()

print(complaint)

From: arun.kumar@example.com
Subject: Broken mixer grinder - need refund

Hi team,

I received my Mixer Grinder (order ORD-1002) yesterday and the main jar is cracked.
It looks like it was damaged during shipping. I have not used the product at all.
I am very disappointed. Please refund my money as soon as possible.

Regards,
Arun Kumar



## Stage 1 — Read and Classify the Complaint

In [9]:
classify_prompt = f"""
You are a customer support triage assistant.

Read the customer message below and return ONLY a JSON object (no extra text) with these keys:
- "category": one of REFUND, DELIVERY, DAMAGED_ITEM, TECHNICAL, BILLING, ACCOUNT, OTHER
- "priority": one of LOW, MEDIUM, HIGH, URGENT
- "sentiment": one of ANGRY, NEUTRAL, HAPPY
- "order_id": the order ID mentioned (like ORD-1234) or null
- "customer_name": the customer's name or null
- "summary": one-sentence summary of the problem

Customer message:
{complaint}
"""
classification = parse_json(llm.invoke(classify_prompt).text)

print("=== STAGE 1: CLASSIFICATION ===")
print(json.dumps(classification, indent=2))

=== STAGE 1: CLASSIFICATION ===
{
  "category": "DAMAGED_ITEM",
  "priority": "HIGH",
  "sentiment": "ANGRY",
  "order_id": "ORD-1002",
  "customer_name": "Arun Kumar",
  "summary": "The customer received a damaged mixer grinder with a cracked jar and is requesting an immediate refund."
}


## Stage 2 — Tools: Check Order and Search Policies

In [10]:
order = check_order(classification.get("order_id"))
policies = search_policies(f"{classification.get('category', '')} {classification.get('summary', '')}")

print("=== STAGE 2A: ORDER INFO ===")
print(json.dumps(order, indent=2))
print()
print("=== STAGE 2B: RELEVANT POLICIES ===")
print(policies)

=== STAGE 2A: ORDER INFO ===
{
  "customer": "Arun Kumar",
  "item": "Mixer Grinder",
  "amount": 4999,
  "status": "Delivered",
  "ordered_on": "2026-11-10",
  "expected_delivery": "2026-11-16",
  "delivered_on": "2026-11-17"
}

=== STAGE 2B: RELEVANT POLICIES ===
## Damaged or Defective Items
If an item arrives damaged or defective, the customer must report it within 7 days of delivery.
Once reported, the customer is offered a free replacement or a full refund, including shipping charges.
No return pickup fee is charged for damaged or defective items.

## Delayed Delivery
If an order is delayed by more than 5 days beyond the expected delivery date, the customer gets a Rs. 200 goodwill coupon.
If an order is delayed by more than 10 days, the customer may cancel for a full refund.
For delayed orders, apologise and share the latest tracking status.

## Escalation Rules
Always escalate when: the customer mentions legal action, consumer court, or media; there is a safety risk (fire, shock

## Stage 3 — Decide (Resolve or Escalate) and Draft a Reply

In [11]:
today = datetime.now().strftime("%Y-%m-%d")
decision_prompt = f"""
You are a senior customer support agent. Today's date is {today}.

Decide whether you can RESOLVE this issue yourself or must ESCALATE it to a human,
using ONLY the company policies and order information below. Do not invent policies.

Return ONLY a JSON object (no extra text) with these keys:
- "decision": "RESOLVE" or "ESCALATE"
- "reason": 1-2 sentences explaining the decision, referring to the policy used
- "action": the concrete action to take (e.g. "Issue full refund of Rs. 4999", "Send to manager for approval")
- "reply": a polite, empathetic email reply to the customer (under 120 words), signed "Support Team".
          If escalating, tell the customer a specialist will contact them within 24 hours.

Customer message:
{complaint}

Classification:
{json.dumps(classification, indent=2)}

Order information:
{json.dumps(order, indent=2) if order else "ORDER NOT FOUND"}

Company policies:
{policies}
"""
decision = parse_json(llm.invoke(decision_prompt).text)

print("=== STAGE 3: DECISION ===")
print("Decision:", decision["decision"])
print("Reason:  ", decision["reason"])
print("Action:  ", decision["action"])
print()
print("--- Draft reply ---")
print(decision["reply"])

=== STAGE 3: DECISION ===
Decision: RESOLVE
Reason:   The customer reported the damaged item within 7 days of delivery, and the order amount of Rs. 4999 is below the Rs. 10,000 escalation threshold.
Action:   Issue a full refund of Rs. 4999 to the original payment method and schedule a free return pickup.

--- Draft reply ---
Dear Arun,

We sincerely apologize for the disappointment and inconvenience caused by receiving a damaged mixer grinder. 

In accordance with our policy, we have initiated a full refund of Rs. 4999 to your original payment method, which should reflect in your account within 5-7 business days. We will also arrange a free return pickup for the damaged item at no cost to you.

Thank you for bringing this to our attention, and we appreciate your patience.

Warm regards,
Support Team


## Stage 4 — Track the Case

In [12]:
cases = []  # in this notebook cases are kept in memory (the Streamlit app saves them to cases.json)

case = {
    "case_id": f"CASE-{len(cases) + 1:04d}",
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "customer": classification.get("customer_name"),
    "order_id": classification.get("order_id"),
    "category": classification.get("category"),
    "priority": classification.get("priority"),
    "decision": decision["decision"],
    "status": "RESOLVED" if decision["decision"] == "RESOLVE" else "OPEN - ESCALATED",
}
cases.append(case)

print("=== STAGE 4: CASE TRACKER ===")
print(json.dumps(case, indent=2))
print()
print("Unresolved cases:", [c["case_id"] for c in cases if c["status"] != "RESOLVED"])

=== STAGE 4: CASE TRACKER ===
{
  "case_id": "CASE-0001",
  "created_at": "2026-11-19 13:19",
  "customer": "Arun Kumar",
  "order_id": "ORD-1002",
  "category": "DAMAGED_ITEM",
  "priority": "HIGH",
  "decision": "RESOLVE",
  "status": "RESOLVED"
}

Unresolved cases: []
